                                   ViDelivInsights: Unified Food Delivery Analytics Project
                               

In [ ]:
#Import Libraries
import pandas as pd
import json
import sqlite3


In [2]:
orders = pd.read_csv("orders.csv")

print("Orders Data")
display(orders.head())
print(orders.columns)


Orders Data


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name'],
      dtype='object')


In [3]:
with open("users.json", "r") as f:
    users_data = json.load(f)

users = pd.DataFrame(users_data)

print("Users Data")
display(users.head())
print(users.columns)


Users Data


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


Index(['user_id', 'name', 'city', 'membership'], dtype='object')


In [4]:
conn = sqlite3.connect("restaurants.db")

with open("restaurants.sql", "r") as f:
    sql_script = f.read()

conn.executescript(sql_script)

restaurants = pd.read_sql_query("SELECT * FROM restaurants;", conn)

print("Restaurants Data")
splay(restaurants.head())
print(restaurants.coludimns)


Restaurants Data


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


Index(['restaurant_id', 'restaurant_name', 'cuisine', 'rating'], dtype='object')


In [5]:
#clean column names
orders.columns = orders.columns.str.strip().str.lower()
users.columns = users.columns.str.strip().str.lower()
restaurants.columns = restaurants.columns.str.strip().str.lower()


In [6]:
#join
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)

print(orders_users.shape)
display(orders_users.head())


(10000, 9)


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [7]:
#left join
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)

print(final_df.shape)
display(final_df.head())


(10000, 12)


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [33]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)
print(" Great Job  Final dataset created!")


 Great Job  Final dataset created!


In [10]:
# Check column names once
print(final_df.columns)


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating'],
      dtype='object')


In [11]:
#data analysis 
gold_df = final_df[final_df['membership'] == 'Gold']

gold_city_revenue = (
    gold_df.groupby('city')['total_amount']
    .sum()
    .sort_values(ascending=False)
)

gold_city_revenue


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [12]:
#Cuisine with highest average order value
final_df.groupby('cuisine')['total_amount'].mean().sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [13]:
#Users whose total orders > ₹1000
user_total = final_df.groupby('user_id')['total_amount'].sum()
count_users = user_total[user_total > 1000].count()
count_users

2544

In [15]:
#Rating range with highest revenue
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0–3.5', '3.6–4.0', '4.1–4.5', '4.6–5.0']

final_df['rating_range'] = pd.cut(
    final_df['rating'],
    bins=bins,
    labels=labels
)

final_df.groupby('rating_range', observed=False)['total_amount'] \
        .sum() \
        .sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
4.1–4.5    1960326.26
3.0–3.5    1881754.57
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [16]:
#Among Gold members, city with highest average order value
gold_df.groupby('city')['total_amount'].mean().sort_values(ascending=False)

city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [17]:
#Cuisine with lowest restaurants but good revenue
rest_count = final_df.groupby('cuisine')['restaurant_id'].nunique()
revenue = final_df.groupby('cuisine')['total_amount'].sum()

pd.DataFrame({
    'restaurant_count': rest_count,
    'revenue': revenue
}).sort_values('restaurant_count')

,restaurant_count,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [18]:
#% of orders by Gold members
gold_orders = len(final_df[final_df['membership'] == 'Gold'])
total_orders = len(final_df)

round((gold_orders / total_orders) * 100)

50

In [28]:
#Restaurant with highest avg order value but < 20 orders
rest_stats = final_df.groupby('restaurant_name_x').agg({
    'total_amount': 'mean',
    'order_id': 'count'
})

rest_stats_filtered = rest_stats[rest_stats['order_id'] < 20] \
                        .sort_values('total_amount', ascending=False)

rest_stats_filtered.head(10)


,total_amount,order_id
restaurant_name_x,,
Hotel Dhaba Multicuisine,1040.222308,13
Sri Mess Punjabi,1029.180833,12
Ruchi Biryani Punjabi,1002.140625,16
Sri Delights Pure Veg,989.467222,18
Classic Kitchen Family Restaurant,973.167895,19
Hotel Dhaba Chinese,973.125556,18
Amma Mess Pure Veg,965.299444,18
Hotel Biryani Pure Veg,964.577692,13
Annapurna Curry House Multicuisine,954.512353,17


In [30]:
options = [
    "Grand Cafe Punjabi",
    "Grand Restaurant South Indian",
    "Ruchi Mess Multicuisine",
    "Ruchi Foods Chinese"
]

check = final_df[final_df['restaurant_name_x'].isin(options)]

result = check.groupby('restaurant_name_x').agg({
    'total_amount': 'mean',
    'order_id': 'count'
}).sort_values('total_amount', ascending=False)

result


,total_amount,order_id
restaurant_name_x,,
Ruchi Mess Multicuisine,851.226250,40
Grand Restaurant South Indian,842.567586,29
Grand Cafe Punjabi,765.409063,32
Ruchi Foods Chinese,686.603158,19


In [20]:
#Combination with highest revenue
final_df.groupby(['membership', 'cuisine'])['total_amount'].sum().sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [21]:
#Quarter with highest revenue
final_df['order_date'] = pd.to_datetime(final_df['order_date'])
final_df['quarter'] = final_df['order_date'].dt.quarter

final_df.groupby('quarter')['total_amount'].sum().sort_values(ascending=False)

quarter
3    2037385.10
4    2018263.66
1    2010626.64
2    1945348.72
Name: total_amount, dtype: float64

In [22]:
#Total orders by Gold
len(final_df[final_df['membership'] == 'Gold'])


4987

In [23]:
#Revenue from Hyderabad
round(final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum())


1889367

In [24]:
#Distinct users
final_df['user_id'].nunique()

2883

In [25]:
#Avg order value for Gold
round(final_df[final_df['membership']=='Gold']['total_amount'].mean(), 2)


797.15

In [26]:
#Orders with rating ≥ 4.5
len(final_df[final_df['rating'] >= 4.5])


3374

In [27]:
#Orders in top Gold city
top_city = gold_city_revenue.index[0]
len(gold_df[gold_df['city'] == top_city])

1337

In [31]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [32]:
import os
os.listdir()


['.cache',
 '.conda',
 '.config',
 '.eclipse',
 '.expo',
 '.idlerc',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.m2',
 '.matplotlib',
 '.mgltools',
 '.node_repl_history',
 '.p2',
 '.pymol',
 '.pymoltimestamp',
 '.streamlit',
 '.sts4',
 '.vscode',
 '3D Objects',
 'ai voice.py',
 'anaconda3',
 'app.py',
 'AppData',
 'Application Data',
 'Contacts',
 'Cookies',
 'Crop_recommendation.csv',
 'customer_shopping_behavior.csv',
 'DataAnalytics project1.ipynb',
 'Desktop',
 'Documents',
 'Downloads',
 'eclipse',
 'eclipse-workspace',
 'eye1.py',
 'Favorites',
 'final_food_delivery_dataset.csv',
 'git',
 'HackothonProject.ipynb',
 'IntelGraphicsProfiles',
 'ligand.pdqpt.pdb',
 'Links',
 'Local Settings',
 'Microsoft',
 'MicrosoftEdgeBackups',
 'Music',
 'My Documents',
 'NetHood',
 'New folder',
 'NTUSER.DAT',
 'ntuser.dat.LOG1',
 'ntuser.dat.LOG2',
 'NTUSER.DAT{5f7f17cd-6c8a-11ec-bd2c-8b76ac0aee78}.TM.blf',
 'NTUSER.DAT{5f7f17cd-6c8a-11ec-bd2c-8b76ac0aee78}.TMContainer000000000000000000